In [4]:
import pandas as pd
import mysql.connector
from datetime import datetime

In [5]:
# -----------------------------
# 1. Connect to MySQL database
# -----------------------------
conn = mysql.connector.connect(
    host='localhost',
    user="root",
    password="meow",
    database="Walmart"
)
cursor = conn.cursor()

In [6]:
# -----------------------------
# 2. Load Dimension CSVs
# -----------------------------
customer_df = pd.read_csv("customer_master_data.csv", index_col=0)
product_df = pd.read_csv("product_master_data.csv", index_col=0)
transaction_df = pd.read_csv("transactional_data.csv", index_col=0)

In [7]:
# -----------------------------
# 3. Populate DimCustomer
# -----------------------------
for _, row in customer_df.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO DimCustomer
        (Customer_ID, Age_Group, Gender, Occupation, City_Category, Stay_In_Current_City_Years, Marital_Status)
        VALUES (%s,%s,%s,%s,%s,%s,%s)
    """, (
        row['Customer_ID'],
        row['Age'],  # Using Age directly; can transform into ranges if needed
        row['Gender'],
        row['Occupation'],
        row['City_Category'],
        str(row['Stay_In_Current_City_Years']),  # store as string
        row['Marital_Status']
    ))
conn.commit()

In [8]:
# -----------------------------
# 4. Populate DimProduct
# -----------------------------
for _, row in product_df.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO DimProduct
        (Product_ID, Product_Category_1, Product_Category_2, Product_Category_3)
        VALUES (%s,%s,%s,%s)
    """, (
        row['Product_ID'],
        row['Product_Category'],  # main category
        None,  # subcategories empty
        None
    ))
conn.commit()

In [9]:
# -----------------------------
# 5. Populate DimDate (from transactional data)
# -----------------------------
transaction_df['date'] = pd.to_datetime(transaction_df['date'])
unique_dates = transaction_df['date'].drop_duplicates()

for dt in unique_dates:
    date_key = int(dt.strftime("%Y%m%d"))
    day_of_week_name = dt.strftime("%A")
    day_of_week_num = dt.isoweekday() % 7 + 1  # MySQL 1=Sun, 7=Sat
    is_weekend = 1 if day_of_week_num in [1, 7] else 0
    month_name = dt.strftime("%B")
    month_num = dt.month
    quarter = (dt.month - 1) // 3 + 1
    year = dt.year

    cursor.execute("""
        INSERT IGNORE INTO DimDate
        (Date_Key, Full_Date, Day_Of_Week_Name, Day_Of_Week_Num, Is_Weekend, Month_Name, Month_Num, Quarter, Year)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        date_key, dt.strftime("%Y-%m-%d"), day_of_week_name, day_of_week_num,
        is_weekend, month_name, month_num, quarter, year
    ))
conn.commit()

In [10]:
# -----------------------------
# 6. Populate FactSales
# -----------------------------
# Merge transactional data with product prices
enriched_df = transaction_df.merge(product_df[['Product_ID','price$']], on='Product_ID', how='left')

for _, row in enriched_df.iterrows():
    date_key = int(pd.to_datetime(row['date']).strftime("%Y%m%d"))
    purchase_amount = row['quantity'] * row['price$']

    cursor.execute("""
        INSERT IGNORE INTO FactSales
        (Transaction_ID, Customer_ID, Product_ID, Date_Key, Purchase_Amount)
        VALUES (%s,%s,%s,%s,%s)
    """, (
        str(row['orderID']),
        row['Customer_ID'],
        row['Product_ID'],
        date_key,
        purchase_amount
    ))

conn.commit()

In [11]:
# -----------------------------
# 7. Close Connection
# -----------------------------
cursor.close()
conn.close()
print("Data Warehouse populated successfully!")

Data Warehouse populated successfully!
